# Notebook 1 - Taxonomy Extraction

**Purpose:** Extract, reconcile and structure the complete category taxonomy from all production rule source files. Output feeds directly into Notebook 2 (LLM enrichment).

**Inputs:**

| File | Description |
|---|---|
| `layered_ns_rules.csv` | Non-spend pattern rules |
| `pattern_matching.csv` | Spend pattern rules |
| `merchant_taxonomy.csv` | Merchant to category lookup |
| `biller_mapping.csv` | BPAY biller to category lookup |
| `mcc_mapping.csv` | MCC to category (`base_category` only) |
| `new_bank_patterns.csv` | Provider-specific overrides with confidence |

**Outputs:**
- `taxonomy_master.csv` - full deduplicated key list with hierarchy
- `taxonomy_master.json` - structured for LLM prompt injection
- `taxonomy_validation.csv` - coverage report against test dataset

## 0. Setup

In [ ]:
import pandas as pd
import json
from pathlib import Path

# --- Paths (update if needed) ---
ASSET_DIR  = Path("assets") 
DATA_DIR   = Path('data')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

FILES = {
    'ns_rules':     ASSET_DIR / 'layered_ns_rules.csv',
    'spend_rules':  ASSET_DIR / 'pattern_matching.csv',
    'merchant':     ASSET_DIR / 'merchant_taxonomy_194.csv',
    'biller':       ASSET_DIR / 'biller_mapping.csv',
    'mcc':          ASSET_DIR / 'mcc_mapping.csv',
    'bank_pattern': ASSET_DIR / 'new_bank_patterns.csv',
    'test_dataset': DATA_DIR / 'masked_transaction_dataset.parquet',
}

# Column in test dataset that holds the assigned category key
# Update if your test dataset uses a different column name
TEST_CAT_COL = 'base_category_key'

# --- Helper: load and report shape ---
def load(name, **kwargs):
    path = FILES[name]
    df   = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path, **kwargs)
    print(f"  {name:20s} {df.shape[0]:>7,} rows  {df.shape[1]:>2} cols")
    return df

print('Loading source files...\n')
ns      = load('ns_rules')
spend   = load('spend_rules')
merch   = load('merchant', low_memory=False)
biller  = load('biller')
mcc     = load('mcc')
bank_pt = load('bank_pattern')
test    = load('test_dataset')

## 1. Inspect Sources

Quick check of the key and type columns we'll use from each file. Adjust column names here if source files change.

In [ ]:
# Column mapping - single place to update if source schemas change
COL_MAP = {
    'ns_rules':     {'key': 'base_category_key', 'type': 'Type'},
    'spend_rules':  {'key': 'new_category_key',  'type': 'new_category_t',
                     'key_ext': 'new_category_ext'},
    'merchant':     {'key': 'category'},
    'biller':       {'key': 'category'},
    'mcc':          {'key': 'base_category'},
    'bank_pattern': {'key': 'category type', 'confidence': 'confidence',
                     'provider': 'provider'},
}

src_dfs = {
    'ns_rules': ns, 'spend_rules': spend, 'merchant': merch,
    'biller': biller, 'mcc': mcc, 'bank_pattern': bank_pt
}

print('Key column samples per source:\n')
for name, cols in COL_MAP.items():
    df     = src_dfs[name]
    k      = cols['key']
    sample = df[k].dropna().unique()[:5].tolist()
    print(f'  {name:20s} "{k}": {sample}')

## 2. Normalise & Extract Category Keys

Normalise known malformed or inconsistently formatted keys across all source
files before extraction. Then pull distinct keys into a common schema:
`(category_key, category_type, is_spend, confidence, provider, source)`

Notes:
- `spend_rules` has two key columns (`new_category_key`, `new_category_ext`) - both extracted
- `mcc_mapping` uses `base_category` only (`vma_base_category` deferred)
- `new_bank_patterns` carries `confidence` and `provider` - retained for later use

Normalise known malformed or inconsistently formatted keys before extraction
Fixes apply across all source files and production data consistently
Add new entries here as issues are discovered

In [ ]:
KEY_NORMALISATION = {
    # Typos
    "FESS":                  "FEES",
    # Spacing variants
    "CREDIT CARD PAYMENT":   "CREDIT_CARD_PAYMENT",
    "CASH WITHDRAWALS":      "CASH_WITHDRAWALS",
    "CHILD MAINTENANCE":     "CHILD_MAINTENANCE",
    "ROUND UP":              "ROUND_UP",
    # Slash variants
    "HEALTHCARE/MEDICAL":    "HEALTHCARE_MEDICAL",
    "REFUNDS/ADJUSTMENTS":   "REFUNDS_ADJUSTMENTS",
    "SALARY/REGULAR INCOME": "SALARY",
}

# Ambiguous keys - no clear mapping, retained for review rather than dropped
# Once resolved, move to KEY_NORMALISATION above
AMBIGUOUS_KEYS = {
    "NOT ROUND UP": "Unclear - opposite of ROUND_UP? Review with data owner",
    "CHEQUES":      "No clear equivalent key - review with data owner",
    "OTHER LOANS":  "Possible variant of LOAN - review with data owner",
}

for df in [ns, spend, merch, biller, mcc, bank_pt]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].replace(KEY_NORMALISATION)

print("Key normalisation applied")
print(f"Ambiguous keys flagged for review: {list(AMBIGUOUS_KEYS.keys())}")

# --- Extract ---
def extract(df, key_col, type_col=None, is_spend=None, source="",
            confidence_col=None, provider_col=None):
    """Normalise one source into the common extraction schema."""
    out = pd.DataFrame()
    out["category_key"]  = df[key_col].astype(str).str.strip().str.upper()
    out["category_type"] = df[type_col].values      if type_col        else None
    out["is_spend"]      = is_spend
    out["confidence"]    = df[confidence_col].values if confidence_col  else None
    out["provider"]      = df[provider_col].values   if provider_col    else None
    out["source"]        = source
    return out.dropna(subset=["category_key"])

parts = [
    extract(ns,      "base_category_key", type_col="Type",
            is_spend=False, source="ns_rules"),
    extract(spend,   "new_category_key",  type_col="new_category_t",
            is_spend=True,  source="spend_rules"),
    extract(spend,   "new_category_ext",  type_col="new_category_t",
            is_spend=True,  source="spend_rules"),
    extract(merch,   "category",
            is_spend=True,  source="merchant_taxonomy"),
    extract(biller,  "category",
            is_spend=True,  source="biller_mapping"),
    extract(mcc,     "base_category",
            is_spend=True,  source="mcc"),
    extract(bank_pt, "category type",
            is_spend=None,  source="bank_patterns",
            confidence_col="confidence", provider_col="provider"),
]

raw = pd.concat(parts, ignore_index=True)
raw = raw[raw["category_key"].str.len() > 0]

print(f"\nTotal key-source pairs : {len(raw):,}")
print(f"Distinct keys (raw)    : {raw['category_key'].nunique():,}")


## 3a. Deduplicate & Reconcile

Collapse to one row per key, resolving conflicts:
- **spend_type:** `conditional` if a key appears in both spend and non-spend sources
- **category_type:** source priority - ns_rules > spend_rules > bank_patterns > majority vote
- **confidence / provider:** retained from bank_patterns where present
- **sources:** list of all files that contributed the key

In [ ]:
# --- spend_type ---
def resolve_spend(vals):
    uniq = set(vals.dropna())
    if True in uniq and False in uniq: return "mixed"       # appears in both spend and non-spend sources
    if True  in uniq: return "spend"
    if False in uniq: return "non_spend"
    return "unknown"                                        # no spend classification in source

spend_map = (
    raw.groupby("category_key")["is_spend"]
       .apply(resolve_spend)
       .rename("spend_type")
)

# --- category_type (source priority) ---
PRIORITY = ["ns_rules", "spend_rules", "bank_patterns"]

def resolve_type(grp):
    for src in PRIORITY:
        vals = grp[grp["source"] == src]["category_type"].dropna()
        if not vals.empty:
            return vals.iloc[0]
    majority = grp["category_type"].dropna()
    return majority.mode().iloc[0] if not majority.empty else None

type_map = (
    raw.groupby("category_key")
       .apply(resolve_type, include_groups=False)
       .rename("category_type")
)

# --- confidence & provider (bank_patterns only) ---
bp_meta = (
    raw[raw["source"] == "bank_patterns"]
       .groupby("category_key")
       .agg(
           confidence=("confidence", "first"),
           provider=("provider", lambda x: sorted(set(x.dropna())))
       )
       .reset_index()
)

# --- source provenance ---
src_map = (
    raw.groupby("category_key")["source"]
       .apply(lambda x: sorted(set(x)))
       .rename("sources")
)

# --- merge into master ---
master = (
    spend_map.to_frame()
             .join(type_map)
             .join(src_map)
             .reset_index()
)

master["source_count"]         = master["sources"].apply(len)
master["is_provider_specific"] = master["sources"].apply(lambda s: s == ["bank_patterns"])
master = master.merge(bp_meta, on="category_key", how="left")
master = master.sort_values(["spend_type", "category_type", "category_key"]).reset_index(drop=True)

print(f"Master taxonomy: {len(master)} unique keys\n")
print(master["spend_type"].value_counts().to_string())

## 3b. Mixed Key Analysis

For keys classified as `mixed`, calculate the proportion of spend vs non-spend
appearances across source files. Helps determine whether a mixed key leans
strongly one way or is genuinely ambiguous.

In [ ]:
mixed_keys = master[master["spend_type"] == "mixed"]["category_key"].tolist()

if not mixed_keys:
    print("No mixed keys found.")
else:
    mixed_raw  = raw[raw["category_key"].isin(mixed_keys)].copy()
    mixed_raw  = mixed_raw.dropna(subset=["is_spend"])

    mixed_dist = (
        mixed_raw.groupby(["category_key", "is_spend"])
                 .size()
                 .unstack(fill_value=0)
                 .rename(columns={True: "spend_count", False: "non_spend_count"})
    )
    mixed_dist["total"]         = mixed_dist["spend_count"] + mixed_dist["non_spend_count"]
    mixed_dist["spend_pct"]     = (mixed_dist["spend_count"]     / mixed_dist["total"] * 100).round(1)
    mixed_dist["non_spend_pct"] = (mixed_dist["non_spend_count"] / mixed_dist["total"] * 100).round(1)
    mixed_dist["leans"]         = mixed_dist["spend_pct"].apply(
        lambda x: "spend" if x >= 70 else ("non_spend" if x <= 30 else "ambiguous")
    )
    mixed_dist = mixed_dist.sort_values("spend_pct", ascending=False)

    print(f"{len(mixed_keys)} mixed keys:\n")
    print(mixed_dist[["spend_count", "non_spend_count", "spend_pct", "non_spend_pct", "leans"]].to_string())
    print(f"\nSummary:")
    print(mixed_dist["leans"].value_counts().to_string())

    master = master.merge(
        mixed_dist[["spend_count", "non_spend_count", "spend_pct", "non_spend_pct", "leans"]].reset_index(),
        on="category_key", how="left"
    )

    # Manual override column for mixed keys - populate in taxonomy_master.csv
    # Valid values: 'spend', 'non_spend', or leave blank to defer to 'leans'
    if "manual_spend_override" not in master.columns:
        master["manual_spend_override"] = None
    master.loc[master["spend_type"] == "mixed", "manual_spend_override"] = (
        master.loc[master["spend_type"] == "mixed", "manual_spend_override"].fillna("")
    )
    print("\nmanual_spend_override column added - populate in taxonomy_master.csv for mixed keys")


## 4. Assign category_type for Remaining Keys

Keys with no type from any source file are resolved via `FALLBACK_TYPES`.
Any key still unresolved after this step is flagged - add it to the dict below.

In [ ]:
# Update this dict when new keys are discovered without an explicit type
FALLBACK_TYPES = {
    "SALARY":                         "BENEFITS",
    "INTEREST_INCOME":                "INTERESTS",
    "DIVIDENDS":                      "INTERESTS",
    "SUPERANNUATION":                 "BENEFITS",
    "RENTAL_INCOME":                  "TRANSFERS",
    "TAX_REFUNDS":                    "TRANSFERS",
    "WORKERS_COMPENSATION":           "BENEFITS",
    "GOVERNMENT_BENEFITS_FAMILY":     "BENEFITS",
    "GOVERNMENT_BENEFITS_UNEMPLOYED": "BENEFITS",
    "GOVERNMENT_BENEFITS_OTHER":      "BENEFITS",
    "BENEFITS":                       "BENEFITS",
    "REFUNDS_ADJUSTMENTS":            "TRANSFERS",
    "DEPOSITS":                       "CASH",
    "HEALTHCARE_MEDICAL":             "BENEFITS",
    "OTHER_INCOME":                   "BENEFITS",
    "SALES_SERVICES_INCOME":          "BENEFITS",
    "ANNUITY":                        "BENEFITS",
    "COMMISSION":                     "BENEFITS",
    "INTERNAL_TRANSFER":              "TRANSFERS",
    "EXTERNAL_TRANSFER_IN":           "TRANSFERS",
    "EXTERNAL_TRANSFER_OUT":          "TRANSFERS",
    "MORTGAGE":                       "TRANSFERS",
    "LOAN":                           "TRANSFERS",
    "CREDIT_CARD_PAYMENT":            "CREDIT_CARD_PAYMENT",
    "CHILD_MAINTENANCE":              "TRANSFERS",
    "TRANSFER":                       "TRANSFERS",
    "SAVINGS":                        "TRANSFERS",
    "ROUND_UP":                       "ROUND_UP",
    "PAY_ON_DEMAND":                  "TRANSFERS",
    "INTEREST_CHARGED":               "INTERESTS",
    "ATM_FEES":                       "FEES",
    "CARD_FEES":                      "FEES",
    "OVERDRAFT_FEES":                 "FEES",
    "TRANSACTION_FEES":               "FEES",
    "OTHER_FEES":                     "FEES",
    "FEES":                           "FEES",
    "INTEREST":                       "INTERESTS",
    "LANDTAX_BODYCORP_STRATA_FEES":   "FEES",
    "CASH_WITHDRAWALS":               "CASH",
    "CASH":                           "CASH",
    "TAX_PAYMENTS":                   "FEES",
    "SECURITIES_TRADES_INVESTMENT":   "TRANSFERS",
    "SECURITIES_TRADES_PROCEEDS":     "TRANSFERS",
    "NOTES":                          "NOTES",
    "UNCATEGORISED":                  "NOTES",
}

# Spend: maps base_category_key to category_group (LLM prompt grouping only)
FALLBACK_GROUPS = {
    "GENERAL_MERCHANDISE":           "Shopping & Retail",
    "CLOTHING_SHOES":                "Shopping & Retail",
    "ELECTRONICS":                   "Shopping & Retail",
    "FURNITURE___HOMEWARE":          "Shopping & Retail",
    "HOBBIES":                       "Shopping & Retail",
    "POSTAGE_SHIPPING":              "Shopping & Retail",
    "TOBACCO":                       "Shopping & Retail",
    "ALCOHOL":                       "Food & Drink",
    "TAKEAWAY___SNACKS":             "Food & Drink",
    "RESTAURANTS__CAFES":            "Food & Drink",
    "GROCERIES":                     "Food & Drink",
    "PUBLIC_TRANSPORT":              "Transport",
    "TAXI___RIDESHARE":              "Transport",
    "FUEL":                          "Transport",
    "PARKING":                       "Transport",
    "CABLE_SATELLITE_TELECOM":       "Bills & Utilities",
    "TELEPHONE_SERVICES":            "Bills & Utilities",
    "SUBSCRIPTIONS_RENEWALS":        "Bills & Utilities",
    "INSURANCE_MEDICAL":             "Bills & Utilities",
    "INSURANCE_OTHER":               "Bills & Utilities",
    "MEDIA_STREAMING":               "Entertainment",
    "GAMBLING_LOTTERIES":            "Entertainment",
    "DEPOSITS_GAMBLING_LOTTERIES":   "Entertainment",
    "HOME_RENOVATION___MAINTENANCE": "Home",
    "EDUCATION_PRIVATE":             "Education",
    "EDUCATION_PUBLIC":              "Education",
    "EDUCATION_TERTIARY":            "Education",
    "CHILDCARE":                     "Family",
    "CHILD_DEPENDENT_EXPENSES":      "Family",
    "ADVERTISING":                   "Business",
    "BUSINESS_MISCELLANEOUS":        "Business",
    "OFFICE_EXPENSES":               "Business",
    "PRINTING":                      "Business",
    "SERVICES_SUPPLIES":             "Business",
    "CHARITABLE_GIVING":             "Other",
    "OTHER_IRREG_EXPENSES":          "Other",
    "OTHER_REG_EXPENSES":            "Other",
    "UNCATEGORISED":                 "System",
}

# Apply category_type to non-spend keys only
master["category_type"] = master.apply(
    lambda r: r["category_type"] if r["spend_type"] == "non_spend" else None,
    axis=1
)
master["category_type"] = master.apply(
    lambda r: r["category_type"] or FALLBACK_TYPES.get(r["category_key"]),
    axis=1
)

# Apply category_group to spend keys only
master["category_group"] = master.apply(
    lambda r: FALLBACK_GROUPS.get(r["category_key"]) if r["spend_type"] == "spend" else None,
    axis=1
)

# Drop parsing artifacts
BAD_KEYS = {"NAN"}
master = master[~master["category_key"].isin(BAD_KEYS)].reset_index(drop=True)

# Mark ambiguous keys as Pending Review
master.loc[master["category_key"].isin(AMBIGUOUS_KEYS), "category_group"] = "Pending Review"

# Flag source file issues for data owner review
KNOWN_ISSUES = {
    "FESS":                "Typo for FEES - fix in source file",
    "CREDIT CARD PAYMENT": "Should be CREDIT_CARD_PAYMENT - fix in source file",
    "CHEQUES":             "No clear mapping - review with data owner",
    "OTHER LOANS":         "Possible variant of LOAN - review with data owner",
}
print("Source file issues to review:")
for k, note in KNOWN_ISSUES.items():
    print(f"  {k}: {note}")

# Report unresolved keys
unresolved_type  = master[(master["spend_type"] == "non_spend") & master["category_type"].isna()]
unresolved_group = master[(master["spend_type"] == "spend")     & master["category_group"].isna()]

if not unresolved_type.empty:
    print(f"\nNon-spend keys missing category_type:\n{unresolved_type['category_key'].tolist()}")
if not unresolved_group.empty:
    print(f"\nSpend keys missing category_group:\n{unresolved_group['category_key'].tolist()}")
if unresolved_type.empty and unresolved_group.empty:
    print(f"\nAll {len(master)} keys resolved")

## 5. Validate Against Test Dataset

Checks:
- **Coverage:** % of test transactions matching a known taxonomy key
- **Missing:** keys in test data not in taxonomy (add to master)
- **Unused:** taxonomy keys not seen in test data (may be rare or deprecated)
- **Distribution:** frequency of each key in test data

In [ ]:
if TEST_CAT_COL not in test.columns:
    print(f"Column '{TEST_CAT_COL}' not found. Available: {test.columns.tolist()}")
else:
    test_keys    = test[TEST_CAT_COL].astype(str).str.strip().str.upper().dropna()
    taxonomy_set = set(master["category_key"])
    test_set     = set(test_keys.unique())

    only_prod = test_set - taxonomy_set
    only_tax  = taxonomy_set - test_set
    coverage  = test_keys.isin(taxonomy_set).mean()

    print(f"Test rows            : {len(test):,}")
    print(f"Distinct keys (test) : {len(test_set)}")
    print(f"Coverage             : {coverage:.1%}")
    print(f"Only in test data    : {len(only_prod)}  <- review & add to taxonomy")
    print(f"Only in taxonomy     : {len(only_tax)}  <- may be rare or deprecated")

    if only_prod:
        print(f"\nKeys in test data not in taxonomy:\n{sorted(only_prod)}")

    dist = (
        test_keys.value_counts()
                 .rename_axis("category_key")
                 .reset_index(name="count")
    )
    dist["pct"]         = (dist["count"] / len(test_keys) * 100).round(2)
    dist["in_taxonomy"] = dist["category_key"].isin(taxonomy_set)

    print(f"\nTop 20 by frequency:\n")
    print(dist.head(20).to_string(index=False))

## 6. Production DB Verification (Optional)

Confirms no keys exist in production that are absent from the taxonomy.
Run once to establish ground truth - not required on every iteration.

Update `DB_CONN` with your connection string before running.

In [ ]:
from dotenv import load_dotenv
import databricks.sql as sql
import os, gzip
load_dotenv(dotenv_path=r"/home/sagemaker-user/swtest1/llm_poc/.env", override=True)

In [ ]:
def dbx_query_df(query: str, *, catalog=None, schema=None) -> pd.DataFrame:
    with sql.connect(
        server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
        http_path=os.getenv("DATABRICKS_HTTP_PATH"),
        access_token=os.getenv("DATABRICKS_TOKEN"),
        catalog=catalog or os.getenv("DATABRICKS_CATALOG"),
        schema=schema  or os.getenv("DATABRICKS_SCHEMA"),
        use_cloud_fetch=True,
    ) as conn, conn.cursor() as cur:
        cur.execute(query)
        return cur.fetchall_arrow().to_pandas(types_mapper=pd.ArrowDtype)


def dbx_scalar(query: str, params=None, *, catalog=None, schema=None):
    with sql.connect(
        server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
        http_path=os.getenv("DATABRICKS_HTTP_PATH"),
        access_token=os.getenv("DATABRICKS_TOKEN"),
        catalog=catalog or os.getenv("DATABRICKS_CATALOG"),
        schema=schema  or os.getenv("DATABRICKS_SCHEMA"),
        use_cloud_fetch=True,
    ) as conn, conn.cursor() as cur:
        cur.execute(query, params or ())
        return cur.fetchone()[0]  # first column of the first row

In [ ]:
RUN_DB_CHECK = True

# --- Config ---
DB_DATE_FROM        = "2025-11-01"
DB_DATE_TO          = "2026-01-31"
TARGET_PER_PROVIDER = 10_000

if RUN_DB_CHECK:
    db_keys_raw = dbx_query_df(f"""
        SELECT
            base_category_key,
            COUNT(*)                AS txn_count,
            MIN(transaction_date)   AS first_seen,
            MAX(transaction_date)   AS last_seen
        FROM (
            SELECT
                t.base_category_key,
                t.transaction_date,
                p.name AS provider_name
            FROM prod.public.accounts_silver_view a
            INNER JOIN prod.public.provider_accounts_silver_view pa ON pa.id = a.provider_account_id
            INNER JOIN prod.public.providers_silver_view p ON p.id = pa.provider_id
            INNER JOIN prod.public.transactions_silver_view t ON t.account_id = a.id
            WHERE p.ideas_enrich_v2 = True
              AND t.base_category_key IS NOT NULL
              AND t.created_at BETWEEN DATE '{DB_DATE_FROM}' AND DATE '{DB_DATE_TO}'
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY p.name
                ORDER BY RAND(42)
            ) <= {TARGET_PER_PROVIDER}
        ) sampled
        GROUP BY base_category_key
        ORDER BY txn_count DESC
    """)

    # 1. Normalise DB keys first - must happen before building db_set
    db_keys_raw["base_category_key"] = (
        db_keys_raw["base_category_key"]
        .str.strip()
        .str.upper()
        .replace(KEY_NORMALISATION)
    )

    # 2. Now build sets for comparison - both sides are normalised
    taxonomy_set = set(master["category_key"])
    db_set       = set(db_keys_raw["base_category_key"])

    only_db  = db_set - taxonomy_set
    only_tax = taxonomy_set - db_set

    print(f"Date range               : {DB_DATE_FROM} → {DB_DATE_TO}")
    print(f"Sample per provider      : {TARGET_PER_PROVIDER:,}")
    print(f"Production distinct keys : {len(db_set)}")
    print(f"Only in production       : {len(only_db)}  <- review & add to taxonomy")
    print(f"Only in taxonomy         : {len(only_tax)} <- may be rare or deprecated")

    if only_db:
        print(f"\nKeys in production not in taxonomy:\n{sorted(only_db)}")

    # 3. Merge DB counts onto master (drop prior run columns first)
    master = master.drop(columns=[c for c in ["txn_count", "first_seen", "last_seen"] if c in master.columns])
    master = master.merge(
        db_keys_raw.rename(columns={"base_category_key": "category_key"}),
        on="category_key", how="left"
    )

else:
    print("DB check skipped - set RUN_DB_CHECK = True to run")

print("\nKeys only in taxonomy (not seen in production) - may be deprecated:")
print(sorted(only_tax))

In [ ]:
print({k: v for k, v in KEY_NORMALISATION.items() if 'CHEQUE' in k or 'CREDIT' in k})

In [ ]:
# Run before the normalisation step to see raw prod keys containing CHEQUE or CREDIT
print(db_keys_raw[db_keys_raw["base_category_key"].str.contains("CHEQUE|CREDIT", na=False)])

## 7. Export

Two formats:
- **CSV** for human review and Notebook 2 input
- **JSON** structured as `spend_type > category_type > [keys]` for LLM prompt injection

In [ ]:
# --- CSV ---
export_cols = [
    "category_key", "spend_type",
    "spend_count", "non_spend_count", "spend_pct", "non_spend_pct",
    "leans", "manual_spend_override",
    "category_type", "category_group",
    "is_provider_specific", "confidence", "provider",
    "source_count", "sources"
]
master[export_cols].to_csv(OUTPUT_DIR / "taxonomy_master.csv", index=False)
print(f"taxonomy_master.csv  ({len(master)} keys)")

if TEST_CAT_COL in test.columns:
    dist.to_csv(OUTPUT_DIR / "taxonomy_validation.csv", index=False)
    print("taxonomy_validation.csv")

# --- JSON hierarchy ---
hierarchy = {"spend": {}, "non_spend": {}, "mixed": {}, "unknown": {}}

for _, row in master.iterrows():
    s = row["spend_type"]
    t = str(row["category_group"] or row["category_type"] or "Unknown")
    k = row["category_key"]
    hierarchy.setdefault(s, {}).setdefault(t, []).append(k)

with open(OUTPUT_DIR / "taxonomy_master.json", "w") as f:
    json.dump(hierarchy, f, indent=2)
print("taxonomy_master.json")

print("\nJSON hierarchy preview:\n")
for s, types in hierarchy.items():
    total = sum(len(v) for v in types.values())
    if total == 0:
        continue
    print(f"  {s} ({total} keys)")
    for t, keys in types.items():
        print(f"    {t}: {len(keys)} keys")